<a href="https://colab.research.google.com/github/muterae/cbc/blob/main/LearningCompanionDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Predictive & Generative Learning companion for 2nd Grade students in Competency Based Curriculum Primary Schools


Group Members:
- James Mathenge
- Joyce Mbiriri
- Mtume Owino

The goal of this notebook is to demonstrate a classifier and adaptive generator for CBC primary school students.


This notebook describes a system that classifies a student into a competency band Below Expectation (BE), Approaching Expectation (AE), Meeting Expectation (ME) and Exceeding Expectation (EE). The student can then generate a quiz that will scale based on their predicted competency band.


In [1]:
# Upgrade scikit-learn to version 1.9.1 (allowing pre-releases if it is a development version)
!pip install --upgrade scikit-learn==1.9.1 --pre

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.8 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


*(Note: After running the pip install command above, please restart your session by going to **Runtime -> Restart session** in the top menu to load the newly installed scikit-learn version into memory, then re-run the loader cell.)*

## Step 1: Load the bundle for classification

#



### Upload and Load your model `.pkl` file
Run the cell below to upload your file directly. Once uploaded, you can load it using `pickle` or `joblib`.

In [2]:
import os
from pathlib import Path
import pickle
from sklearn.pipeline import Pipeline
import joblib
from google.colab import files


# Define the target filename we expect
pkl_filename = 'cbc_predictor_bundle.pkl'

# List of potential paths to look for the file (e.g., current directory, or common subfolders if cloned from GitHub)
potential_paths = [
    pkl_filename,                             # Current working directory (Colab root /content/ or repo root)
    os.path.join('models', pkl_filename),     # Common subfolder in GitHub repositories
    os.path.join('data', pkl_filename)        # Another common subfolder
]

file_found = False
for path in potential_paths:
    if os.path.exists(path):
        pkl_filename = path
        file_found = True
        print(f"Found model bundle locally at: {pkl_filename}")
        break

# If not found locally, prompt the user to upload it
if not file_found:
    print(f"'{pkl_filename}' not found in local workspace paths.")
    print("Please upload the model bundle file:")
    uploaded = files.upload()
    uploaded_file_names = list(uploaded.keys())
    if uploaded_file_names:
        pkl_filename = uploaded_file_names[0]
        file_found = True
        print(f"Successfully uploaded: {pkl_filename}")

# Load the model bundle if available
if file_found:
    try:
        path = Path(pkl_filename)
        bundle = joblib.load(path)
        print("Bundle loaded successfully!")
    except Exception as e:
        print(f"Error loading the bundle file: {e}")
else:
    print("Model bundle could not be found or loaded.")

'cbc_predictor_bundle.pkl' not found in local workspace paths.
Please upload the model bundle file:


Saving cbc_predictor_bundle.pkl to cbc_predictor_bundle.pkl
Successfully uploaded: cbc_predictor_bundle.pkl
Bundle loaded successfully!


# Step 2: Validation
We must confirm that the bundle has loaded correctly with all the required keys and labels.


In [3]:
# check the bundle for missing required keys
required_keys = {
        "pipeline",
        "label_encoder",
        "expected_columns",
        "band_cut_points",
        "metrics",
  }
missing_keys = required_keys - set(bundle)
if missing_keys:
    raise KeyError(f"Bundle is missing keys: {sorted(missing_keys)}")



VALID_BANDS = {
    "Below Expectation",
    "Approaching Expectation",
    "Meeting Expectation",
    "Exceeding Expectation",
}

encoder_labels = list(bundle["label_encoder"].classes_)
if set(encoder_labels) != VALID_BANDS:
    raise AssertionError(f"Encoder classes are not the four CBC bands: {encoder_labels}")

print(f"Model trained at: {bundle['metrics']['trained_at']}")
print(
    "Cross-validation accuracy: "
    f"{bundle['metrics']['cross_validation']['accuracy_mean']:.3f}"
)
print(f"Encoder classes: {encoder_labels}\n")

Model trained at: 2026-10-29T09:58:37.120287+00:00
Cross-validation accuracy: 0.742
Encoder classes: ['Approaching Expectation', 'Below Expectation', 'Exceeding Expectation', 'Meeting Expectation']



# Step 3: Create student profile
In this step, we generate three different students.

`student_profile_1` - describes a strong learner which should be classified as ME or higher

`student_profile_2` - describes an average learner which should be classified between ME and AE.

`student_profile_3` - describes a struggling user which should be classified as ME or lower.


These students will server as our input and we will use them thorughout the rest of the notebook.

In [4]:
SAMPLE_LEARNERS = {
    "student_profile_1": {
        "Hours_Studied": 30,
        "Attendance": 95,
        "Sleep_Hours": 8,
        "Previous_Scores": 90,
        "Tutoring_Sessions": 4,
        "Physical_Activity": 5,
        "Parental_Involvement": "High",
        "Access_to_Resources": "High",
        "Extracurricular_Activities": "Yes",
        "Motivation_Level": "High",
        "Internet_Access": "Yes",
        "Family_Income": "High",
        "Teacher_Quality": "High",
        "School_Type": "Private",
        "Peer_Influence": "Positive",
        "Learning_Disabilities": "No",
        "Parental_Education_Level": "College",
        "Distance_from_Home": "Near",
        "Gender": "Female",
    },
    "student_profile_2": {
        "Hours_Studied": 20,
        "Attendance": 80,
        "Sleep_Hours": 7,
        "Previous_Scores": 72,
        "Tutoring_Sessions": 2,
        "Physical_Activity": 3,
        "Parental_Involvement": "Medium",
        "Access_to_Resources": "Medium",
        "Extracurricular_Activities": "Yes",
        "Motivation_Level": "Medium",
        "Internet_Access": "Yes",
        "Family_Income": "Medium",
        "Teacher_Quality": "Medium",
        "School_Type": "Public",
        "Peer_Influence": "Neutral",
        "Learning_Disabilities": "No",
        "Parental_Education_Level": "High School",
        "Distance_from_Home": "Moderate",
        "Gender": "Male",
    },
    "student_profile_3": {
        "Hours_Studied": 8,
        "Attendance": 60,
        "Sleep_Hours": 6,
        "Previous_Scores": 55,
        "Tutoring_Sessions": 0,
        "Physical_Activity": 1,
        "Parental_Involvement": "Low",
        "Access_to_Resources": "Low",
        "Extracurricular_Activities": "No",
        "Motivation_Level": "Low",
        "Internet_Access": "No",
        "Family_Income": "Low",
        "Teacher_Quality": "Low",
        "School_Type": "Public",
        "Peer_Influence": "Negative",
        "Learning_Disabilities": "No",
        "Parental_Education_Level": "High School",
        "Distance_from_Home": "Far",
        "Gender": "Male",
    },
}


student_profile_1 = SAMPLE_LEARNERS["student_profile_1"]
student_profile_2 = SAMPLE_LEARNERS["student_profile_2"]
student_profile_3 = SAMPLE_LEARNERS["student_profile_3"]

# Step 4: Classify the students
Using the student profiles paired with a `predict` function, a list of all the bands with a percentage of how much each student relates to it will be generated.


`predict` - Predicts one learner and return the band plus class probabilities.

In [5]:
import pandas as pd

def predict(artifact: dict, learner: dict) -> tuple[str, dict[str, float]]:
    missing = set(artifact["expected_columns"]) - set(learner)
    if missing:
        raise ValueError(f"Learner payload is missing columns: {sorted(missing)}")

    row = pd.DataFrame([learner])[artifact["expected_columns"]]

    pipeline = artifact["pipeline"]
    encoder = artifact["label_encoder"]

    index = int(pipeline.predict(row)[0])
    band = encoder.inverse_transform([index])[0]

    probabilities = {
        label: float(prob)
        for label, prob in zip(encoder.classes_, pipeline.predict_proba(row)[0])
    }
    return band, probabilities




failures = 0
for name, learner in SAMPLE_LEARNERS.items():
    band, probabilities = predict(bundle, learner)
    print(f"{name}: {band}")
    sorted_probalities = sorted(probabilities.items(), key=lambda item: -item[1])
    for label, prob in sorted_probalities:
        print(f"    {label:<24} {prob * 100:5.1f}%")

    SAMPLE_LEARNERS[name]['band'] = sorted_probalities[0]

    if band not in VALID_BANDS:
        print(f"  FAIL: '{band}' is not a valid CBC band")
        failures += 1
    if abs(sum(probabilities.values()) - 1.0) > 1e-6:
        print("  FAIL: probabilities do not sum to 1")
        failures += 1
    print()

if failures:
    print(f"{failures} check(s) failed.")


print("All inference checks passed.")



student_profile_1: Exceeding Expectation
    Exceeding Expectation    100.0%
    Meeting Expectation        0.0%
    Below Expectation          0.0%
    Approaching Expectation    0.0%

student_profile_2: Meeting Expectation
    Meeting Expectation       42.9%
    Approaching Expectation   38.3%
    Below Expectation         16.8%
    Exceeding Expectation      2.0%

student_profile_3: Below Expectation
    Below Expectation        100.0%
    Approaching Expectation    0.0%
    Exceeding Expectation      0.0%
    Meeting Expectation        0.0%

All inference checks passed.


# Step 5: Create the corpus for generating quizzes
Now that we have near-accurately classified students to competency bands, we have to instruct a RAG on how to generate quizzes based on a give band. This is done by creating a `band_playbook`.

Simply put, it is a rule book for the RAG. By defining what to include and exclude, the RAG will be able to effectively generate compatible quizzes for each band.



In [6]:
from enum import StrEnum
from pydantic import BaseModel, Field, field_validator, model_validator


class Band(StrEnum):
    BE = "Below Expectation"
    AE = "Approaching Expectation"
    ME = "Meeting Expectation"
    EE = "Exceeding Expectation"

class QuizItem(BaseModel):
    """A single four-option item in the shape the mobile client expects."""

    question_text: str = Field(min_length=1)
    options: list[str] = Field(min_length=4, max_length=4)
    correct_answer: str
    hint: str = Field(min_length=1)

    @model_validator(mode="after")
    def check_options(self) -> "QuizItem":
        if len(set(self.options)) != 4:
            raise ValueError("options must be unique")
        if self.correct_answer not in self.options:
            raise ValueError("correct_answer must be one of options")
        return self


class BandPlaybook(BaseModel):
    code: Band
    rank: int = Field(ge=1, le=4)
    question_count: int = Field(ge=1)
    max_question_words: int = Field(ge=1)
    number_range: tuple[int, int]
    regrouping: bool
    max_steps: int = Field(ge=1)
    cognitive_level: str
    task_style: str
    context_style: str
    hint_style: str
    distractor_policy: str
    description: str
    prompt_directives: list[str] = Field(min_length=1)
    forbidden: list[str] = Field(min_length=1)
    teacher_insight_template: str
    example_item: QuizItem


BAND_PLAYBOOK: dict[Band, BandPlaybook] = {
    Band.BE: BandPlaybook(
        code=Band.BE,
        rank=1,
        question_count=4,
        max_question_words=12,
        number_range=(0, 10),
        regrouping=False,
        max_steps=1,
        cognitive_level="Recognise and count with concrete support",
        task_style="Direct counting or matching with objects, pictures or counters.",
        context_style="Classroom counters, sticks, bottle tops and learners.",
        hint_style="Step-by-step physical action, e.g. touch each object while counting.",
        distractor_policy="Options visibly different in size; avoid near-misses.",
        description=(
            "Learner needs concrete support to recognise numbers and complete "
            "one-step counting tasks."
        ),
        prompt_directives=[
            "Use numbers from 0 to 10 only.",
            "Use concrete objects or pictures described in words.",
            "Ask one direct question with one action.",
            "Keep the question at or under 12 words.",
            "Make the correct option clearly different from the distractors.",
            "Give a hint that tells the learner exactly what to do physically.",
        ],
        forbidden=[
            "No carrying or borrowing (regrouping).",
            "No multi-step word problems.",
            "No missing-number or inverse reasoning.",
            "No abstract symbols without a concrete context.",
            "No negative numbers.",
        ],
        teacher_insight_template=(
            "Learner is practising counting and number recognition with "
            "concrete support."
        ),
        example_item=QuizItem(
            question_text="There are 4 mangoes. How many mangoes are there?",
            options=["A) 2", "B) 3", "C) 4", "D) 5"],
            correct_answer="C) 4",
            hint="Touch each mango as you count: one, two, three, four.",
        ),
    ),
    Band.AE: BandPlaybook(
        code=Band.AE,
        rank=2,
        question_count=5,
        max_question_words=20,
        number_range=(0, 20),
        regrouping=False,
        max_steps=1,
        cognitive_level="Apply one familiar procedure",
        task_style="One-step arithmetic or a short word problem with familiar realia.",
        context_style="Kenyan everyday objects: mangoes, shillings, sukuma wiki, pencils.",
        hint_style="Rule reminder, e.g. count on from the larger number.",
        distractor_policy="One off-by-one near-miss; the other options are clearly wrong.",
        description=(
            "Learner can apply a single familiar procedure with numbers up to 20 "
            "when the context is familiar."
        ),
        prompt_directives=[
            "Use numbers from 0 to 20 only.",
            "Use one operation and one step.",
            "Use familiar Kenyan objects or money.",
            "Do not require carrying or borrowing.",
            "Keep the question at or under 20 words.",
            "Include one near-miss distractor.",
        ],
        forbidden=[
            "No carrying or borrowing (regrouping).",
            "No multi-step word problems.",
            "No missing-number or inverse reasoning.",
            "No negative numbers.",
            "No unfamiliar or abstract contexts.",
        ],
        teacher_insight_template=(
            "Learner is applying single-step arithmetic in familiar everyday contexts."
        ),
        example_item=QuizItem(
            question_text=(
                "Amina has 12 mangoes. She buys 5 more. "
                "How many mangoes does she have now?"
            ),
            options=["A) 15", "B) 16", "C) 17", "D) 18"],
            correct_answer="C) 17",
            hint="Start at 12 and count 5 steps forward.",
        ),
    ),
    Band.ME: BandPlaybook(
        code=Band.ME,
        rank=3,
        question_count=5,
        max_question_words=30,
        number_range=(0, 100),
        regrouping=True,
        max_steps=2,
        cognitive_level="Apply standard Grade 3 procedures",
        task_style="KICD end-of-sub-strand tasks; one- and two-step word problems.",
        context_style="Familiar school and home contexts with Kenyan realia.",
        hint_style="Self-checking prompt, e.g. identify the operation and estimate the answer.",
        distractor_policy="Include a common misconception distractor, e.g. wrong operation or off-by-one.",
        description=(
            "Learner can apply Grade 3 KICD skills to standard one- and two-step tasks."
        ),
        prompt_directives=[
            "Use numbers from 0 to 100.",
            "Allow carrying and borrowing.",
            "Use one or two steps.",
            "Use a KICD-style word problem.",
            "Keep the question at or under 30 words.",
            "Include a distractor based on a common misconception.",
            "Hint should help the learner check the operation or estimate the answer.",
        ],
        forbidden=[
            "No inverse or missing-number reasoning.",
            "No more than two steps.",
            "No negative numbers.",
            "No ambiguous wording.",
            "No content beyond the Grade 3 KICD sub-strand.",
        ],
        teacher_insight_template=(
            "Learner is applying Grade 3 skills to standard one- and two-step tasks."
        ),
        example_item=QuizItem(
            question_text="A pencil costs 5 shillings. How much do 4 pencils cost?",
            options=["A) 9", "B) 15", "C) 20", "D) 25"],
            correct_answer="C) 20",
            hint="Multiply the cost of one pencil by the number of pencils. Check by adding 5 four times.",
        ),
    ),
    Band.EE: BandPlaybook(
        code=Band.EE,
        rank=4,
        question_count=6,
        max_question_words=40,
        number_range=(0, 100),
        regrouping=True,
        max_steps=3,
        cognitive_level="Analyse, reason and transfer",
        task_style=(
            "Inverse reasoning, missing-number or missing-operator problems, "
            "comparison and multi-step tasks."
        ),
        context_style="Less familiar or novel contexts that require transfer.",
        hint_style="Strategy nudge, e.g. work backwards or test each option.",
        distractor_policy="Include a near-miss based on the inverse error; all options plausible.",
        description=(
            "Learner can reason with inverse relationships, missing operators and "
            "multi-step problems."
        ),
        prompt_directives=[
            "Use numbers from 0 to 100.",
            "Allow carrying and borrowing.",
            "Include inverse reasoning, a missing number or operator, or a multi-step problem.",
            "Use a less familiar context to test transfer.",
            "Keep the question at or under 40 words.",
            "Include plausible distractors that reflect common reasoning errors.",
            "Hint should suggest a strategy, not the answer.",
        ],
        forbidden=[
            "No negative numbers.",
            "No algebra with letters.",
            "No content beyond the Grade 3 KICD sub-strand.",
            "No trick questions.",
            "No more than three steps.",
        ],
        teacher_insight_template=(
            "Learner is reasoning with inverse and multi-step problems and "
            "explaining their thinking."
        ),
        example_item=QuizItem(
            question_text="What number goes in the box? \u25a1 + 6 = 15",
            options=["A) 7", "B) 8", "C) 9", "D) 10"],
            correct_answer="C) 9",
            hint="Work backwards: subtract 6 from 15.",
        ),
    ),
}

# Step 6: Define the system prompt
This will be prompt the RAG receives to generate the quiz

In [7]:
from pydantic import Field, field_validator, model_validator
from typing import Literal
from dataclasses import field
import json


class AdaptiveQuizRequest(BaseModel):
    student_id: str = Field(min_length=1)
    strand: str = "Mathematics"
    sub_strand: str | None = None
    session_type: Literal["placement", "practice", "assessment"] = "practice"
    # Optional only for tests/admin tooling. The live path reads the band from
    # the stored learner profile.
    competency_band: str | None = None

class CurriculumLocation(BaseModel):
    strand: str | None
    sub_strand: str | None
    display: str

class RetrievedLesson(BaseModel):
    chunk_id: str
    text: str
    score: float = 0.0
    metadata: dict = field(default_factory=dict)

def format_lesson(lesson: RetrievedLesson, number: int) -> str:
    meta = lesson.metadata or {}
    title = meta.get("lesson_title") or meta.get("section") or lesson.chunk_id
    position = " ".join(
        part
        for part in (
            str(meta.get("term", "")),
            f"W{meta.get('week')}" if meta.get("week") else "",
            f"L{meta.get('lesson')}" if meta.get("lesson") else "",
        )
        if part
    )
    heading = f"[{number}] {lesson.chunk_id} | {title}"
    if position:
        heading += f" | {position}"
    return f"{heading}\n{lesson.text[:1800]}"

def build_messages(
    request: AdaptiveQuizRequest,
    location: CurriculumLocation,
    playbook: BandPlaybook,
    lessons: list[RetrievedLesson],
) -> list[dict]:
    low, high = playbook.number_range
    example = json.dumps(
        playbook.example_item.model_dump(), ensure_ascii=False, indent=2
    )
    directives = "\n".join(f"- {rule}" for rule in playbook.prompt_directives)
    forbidden = "\n".join(f"- {rule}" for rule in playbook.forbidden)
    excerpts = "\n\n".join(
        format_lesson(lesson, number)
        for number, lesson in enumerate(lessons, start=1)
    )

    system = f"""You write Grade 3 Kenyan CBC Mathematics quizzes.

Return ONLY a JSON object. No markdown, no code fences, no explanation.
The JSON shape must be:
{{
  "questions": [
    {{
      "question_text": "string",
      "options": ["A) string", "B) string", "C) string", "D) string"],
      "correct_answer": "A) string",
      "hint": "string",
      "pedagogical_objective": "string"
    }}
  ],
  "teacher_insight_summary": "string"
}}

Hard rules:
- Write exactly {playbook.question_count} questions.
- Every question has exactly 4 unique options, prefixed "A) ", "B) ", "C) " and "D) ".
- correct_answer must exactly match one of the options.
- Use only the curriculum excerpts and the band rules below.
- Keep each question at or under {playbook.max_question_words} words.
- Keep the language simple and suitable for a Grade 3 learner in Kenya.
"""

    user = f"""Requested strand: {location.display}
Competency band: {playbook.code.value} ({playbook.code.name})
Session type: {request.session_type}
Number range: {low} to {high}
Regrouping allowed: {"yes" if playbook.regrouping else "no"}
Maximum steps: {playbook.max_steps}
Maximum words per question: {playbook.max_question_words}

Band directives:
{directives}

Forbidden:
{forbidden}

Example item:
{example}

Curriculum excerpts:
{excerpts}

Create the quiz now."""
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

quiz_request = AdaptiveQuizRequest(
    student_id="188916",
    strand="Mathematics",
    sub_strand=None,
    session_type = "practice",
    competency_band="EE"
)
curriculum = CurriculumLocation(
    strand="Mathematics",
    sub_strand=None,
    display="Mathematics",
)
retrieved_lessons =  []


prompt = build_messages(quiz_request, curriculum, BAND_PLAYBOOK[Band.EE], retrieved_lessons)
print(json.dumps(prompt, indent=5))


[
     {
          "role": "system",
          "content": "You write Grade 3 Kenyan CBC Mathematics quizzes.\n\nReturn ONLY a JSON object. No markdown, no code fences, no explanation.\nThe JSON shape must be:\n{\n  \"questions\": [\n    {\n      \"question_text\": \"string\",\n      \"options\": [\"A) string\", \"B) string\", \"C) string\", \"D) string\"],\n      \"correct_answer\": \"A) string\",\n      \"hint\": \"string\",\n      \"pedagogical_objective\": \"string\"\n    }\n  ],\n  \"teacher_insight_summary\": \"string\"\n}\n\nHard rules:\n- Write exactly 6 questions.\n- Every question has exactly 4 unique options, prefixed \"A) \", \"B) \", \"C) \" and \"D) \".\n- correct_answer must exactly match one of the options.\n- Use only the curriculum excerpts and the band rules below.\n- Keep each question at or under 40 words.\n- Keep the language simple and suitable for a Grade 3 learner in Kenya.\n"
     },
     {
          "role": "user",
          "content": "Requested strand: Mathe

# Step 7: Generate the quiz
The server is already deployed at `https://cbc-099068d6.fastapicloud.dev/`, so all we need to do is make a request to the endpoint `/adaptive-quiz`.

Using the prompt we made above, the RAG will use the already embedded KICD CBC Grade 3 Mathematics document to generate a quiz based on the student learner profile.

In [9]:
import requests
import json

BASE = "https://cbc-099068d6.fastapicloud.dev"

# Loop through each profile in SAMPLE_LEARNERS
for profile_name, learner_data in SAMPLE_LEARNERS.items():
    predicted_band_tuple = learner_data.get('band')
    band_mapping = {
        "Exceeding Expectation": "EE",
        "Meeting Expectation": "ME",
        "Approaching Expectation": "AE",
        "Below Expectation": "BE"
    }

    full_band_name = predicted_band_tuple[0] if predicted_band_tuple else "ME"
    band_code = band_mapping.get(full_band_name, "ME")

    print(f"==================================================")
    print(f"Generating Quiz for: {profile_name}")
    print(f"Predicted Band: {full_band_name} ({band_code})")
    print(f"==================================================\n")

    try:
        r = requests.post(
            f"{BASE}/api/v1/adaptive-quiz",
            json={
                "student_id": "std_ke_09823",
                "strand": "Mathematics",
                "session_type": "practice",
                "competency_band": band_code,
            },
            timeout=180,
        )

        if r.status_code == 200:
            quiz_data = r.json()
            print(f"Teacher Summary:\n{quiz_data.get('teacher_insight_summary')}\n")

            for idx, q in enumerate(quiz_data.get('questions', []), 1):
                print(f"Q{idx}: {q['question_text']}")
                print(f"Options: {', '.join(q['options'])}")
                print(f"Correct: {q['correct_answer']}")
                print(f"Hint: {q['hint']}\n")
        else:
            print(f"Failed to generate quiz. Status code: {r.status_code}\n")

    except Exception as e:
        print(f"An error occurred: {e}\n")


Generating Quiz for: student_profile_1
Predicted Band: Exceeding Expectation (EE)

Teacher Summary:
These items reinforce division up to 90, use multiplication tables, and include word contexts to promote transfer. They balance inverse reasoning, missing numbers, and multi‑step problems while keeping language simple for Grade 3 learners.

Q1: A basket holds 48 apples. If 6 friends share them equally, how many apples does each friend get?
Options: A) 8, B) 6, C) 7, D) 9
Correct: A) 8
Hint: Divide 48 by 6 step by step.

Q2: There are 63 marbles. 9 kids take turns taking 7 marbles each. How many marbles are left?
Options: A) 0, B) 6, C) 9, D) 3
Correct: B) 6
Hint: Subtract 7 nine times or use 63÷7 then multiply by 9.

Q3: A school has 84 students. They are divided into 7 groups for a project. How many students per group?
Options: A) 12, B) 10, C) 14, D) 9
Correct: A) 12
Hint: Think of 84 ÷ 7 using the multiplication table.

Q4: There are 56 stickers. If each child gets 8 stickers, how man

### Workspace & API Resources

- **Model Bundle File**: `/content/cbc_predictor_bundle.pkl` (Loaded successfully)
  - **Model Trained At**: 2026-10-29T09:58:37 UTC
  - **Cross-Validation Accuracy**: 74.2%
  - **Target Classes**: `Below Expectation`, `Approaching Expectation`, `Meeting Expectation`, `Exceeding Expectation`
- **Production API Endpoint**: `https://cbc-099068d6.fastapicloud.dev`
  - **Route**: `POST /api/v1/adaptive-quiz` (Generates custom assessment questions matching the student's predicted competency band)